In [ ]:
%cd /content
!rm -rf worldindex
!git clone https://github.com/RivaanRanawat/worldindex.git worldindex
%cd /content/worldindex

In [ ]:
!curl -sSL https://install.python-poetry.org | python3 -
import os
os.environ["PATH"] += ":/root/.local/bin"
!poetry install --with video

In [ ]:
%%bash

export WORLDINDEX_REAL_EXTRACTION_CLIP_LIMIT=100
poetry run python scripts/pipeline.py extract --config config/gpu_pipeline_droid.yaml


In [ ]:
!poetry run python scripts/pipeline.py run --config config/gpu_pipeline_droid.yaml

In [ ]:
%%bash
cd /content/worldindex

PORT=8000

OLD_PID=$(lsof -ti tcp:${PORT} -sTCP:LISTEN 2>/dev/null | head -n 1 || true)
if [ -n "${OLD_PID}" ]; then
  kill "${OLD_PID}" 2>/dev/null || true
  sleep 2
fi

rm -f /content/worldindex/server.pid

nohup poetry run python scripts/pipeline.py serve --config config/gpu_pipeline_droid.yaml \
  > /content/worldindex/server.log 2>&1 &

echo $! > /content/worldindex/server.pid
sleep 5

echo "server pid: $(cat /content/worldindex/server.pid)"
tail -n 40 /content/worldindex/server.log

In [ ]:
!curl -s http://127.0.0.1:8000/health

In [ ]:
%%bash
cd /content/worldindex
mkdir -p tmp

curl -L "https://datasets-server.huggingface.co/assets/lerobot/aloha_sim_transfer_cube_scripted_image/--/5a682be7b4f6451ed450f84d3bbbe299557c1098/--/default/train/2/observation.images.top/image.jpg?Expires=1772487635&Signature=CapanHiD-RugCiOBEkp2pyKu-MosUHuBHVDJlLfErjG55JYU6ffuclKC-MER7zI8mf4r0nKnY8DtJdnCbrbespJeNTkeEZ2TB852FOOPQVheI9bxnK7SNDYOxKoDHz9X8ciWiKpAkAAFlywku-80jQtf-tiKBTUqCWC3PpwWKbjYSmNlkINAyEMjzFwRTvzKKRrViJm8337eIgL8YL-h4g-pVjRwnE2hVFMX0p841qtkSCH4StSFvEXDxn4VeeUYqYH2fcDY8I1grPX0AU3BxUqbY9LuzTp6LBwsPAUtCij75-zatU4O9ijAQVXRQ4l1TDNSG7qdapP7cQVvUqm-pg__&Key-Pair-Id=K3EI6M078Z3AC3" \
  -o tmp/transfer_cube.jpg

ls -lh tmp/transfer_cube.jpg

In [ ]:
%%bash
cd /content/worldindex
curl -s -F "image=@tmp/transfer_cube.jpg;type=image/jpeg" \
  "http://127.0.0.1:8000/search/image?top_k=5&coarse_k=10" | python -m json.tool